In [25]:
import math
import random

# Data Contoh Pelanggan: (ID, koordinat_x, koordinat_y, demand/kapasitas)
kantor_pusat = (0, 0)
data_pelanggan = [
    (1, 2, 3, 10), (2, 5, 1, 15), (3, -1, 4, 20), (4, 3, -2, 5),
    (5, -3, -2, 12), (6, 4, 4, 8), (7, -2, 2, 10), (8, 0, 5, 15)
]
KAPASITAS_MAX = 35
JUMLAH_KENDARAAN = 3

# 5. Fungsi jarakEuclidean
def jarakEuclidean(a, b):
    return math.sqrt((b[0] - a[0])**2 + (b[1] - a[1])**2)

# 6. Fungsi cariPelanggan
def cariPelanggan(customerID, arrCust):
    for cust in arrCust:
        if cust[0] == customerID:
            return cust
    return None

# 7. Fungsi totalJarak
def totalJarak(routes, pusat, arrCust):
    total = 0.0
    for rute in routes:
        if not rute:
            continue
        # Dari pusat ke pelanggan pertama
        p1 = cariPelanggan(rute[0], arrCust)
        total += jarakEuclidean(pusat, (p1[1], p1[2]))
        
        # Antar pelanggan di dalam rute
        for i in range(len(rute) - 1):
            pa = cariPelanggan(rute[i], arrCust)
            pb = cariPelanggan(rute[i+1], arrCust)
            total += jarakEuclidean((pa[1], pa[2]), (pb[1], pb[2]))
            
        # Dari pelanggan terakhir kembali ke pusat
        p_akhir = cariPelanggan(rute[-1], arrCust)
        total += jarakEuclidean((p_akhir[1], p_akhir[2]), pusat)
    return total

# 8. Fungsi validateRoute
def validateRoute(routes, arrCust, kapasitasMax):
    for rute in routes:
        total_beban = 0
        for custID in rute:
            cust = cariPelanggan(custID, arrCust)
            total_beban += cust[3]
        if total_beban > kapasitasMax:
            return False
    return True

# 9. Fungsi inisiasiSolusiAwal
def inisiasiSolusiAwal(arrCust, jmlKendaraan, kapasitasMax):
    custIDs = [c[0] for c in arrCust]
    while True:
        random.shuffle(custIDs)
        routes = [[] for _ in range(jmlKendaraan)]
        
        # Distribusikan secara acak/berurutan ke kendaraan yang muat
        for cID in custIDs:
            assigned = False
            for rute in routes:
                # Cek sementara jika ditambahkan ke rute ini
                rute.append(cID)
                if validateRoute(routes, arrCust, kapasitasMax):
                    assigned = True
                    break
                else:
                    rute.pop() # batalkan jika overload
            if not assigned:
                break # Gagal membuat rute valid, looping ulang shuffle
        else:
            return routes

# 10. Fungsi cariTetangga
def cariTetanggaSA(routes, arrCust, kapasitasMax):
    routes_baru = [rute.copy() for rute in routes]
    # Pilih 2 rute berbeda secara acak
    idx1, idx2 = random.sample(range(len(routes_baru)), 2)
    
    if routes_baru[idx1] and routes_baru[idx2]:
        # Tukar satu pelanggan acak dari masing-masing rute
        i1 = random.randint(0, len(routes_baru[idx1]) - 1)
        i2 = random.randint(0, len(routes_baru[idx2]) - 1)
        
        routes_baru[idx1][i1], routes_baru[idx2][i2] = routes_baru[idx2][i2], routes_baru[idx1][i1]
        
        if validateRoute(routes_baru, arrCust, kapasitasMax):
            return routes_baru
    return routes # kembalikan rute lama jika tidak valid/kosong

# 11. Implementasi dan Menjalankan simulated_annealing()
def simulated_annealing():
    solusi_sekarang = inisiasiSolusiAwal(data_pelanggan, JUMLAH_KENDARAAN, KAPASITAS_MAX)
    biaya_sekarang = totalJarak(solusi_sekarang, kantor_pusat, data_pelanggan)
    
    T = 1000.0
    alpha = 0.95
    T_min = 0.01
    
    while T > T_min:
        solusi_baru = cariTetanggaSA(solusi_sekarang, data_pelanggan, KAPASITAS_MAX)
        biaya_baru = totalJarak(solusi_baru, kantor_pusat, data_pelanggan)
        
        delta = biaya_baru - biaya_sekarang
        
        if delta < 0 or random.random() < math.exp(-delta / T):
            solusi_sekarang = solusi_baru
            biaya_sekarang = biaya_baru
            
        T *= alpha
        
    print("\n--- SIMULATED ANNEALING SELESAI ---")
    print(f"Total Jarak Akhir: {biaya_sekarang:.2f}")
    print("Rute Kendaraan Akhir:", solusi_sekarang)

simulated_annealing()


--- SIMULATED ANNEALING SELESAI ---
Total Jarak Akhir: 46.66
Rute Kendaraan Akhir: [[8, 3], [7, 1, 6, 4], [5, 2]]


In [26]:
import random

# Data Pendukung untuk Inisialisasi
mapel_list = [f"Mapel_{i}" for i in range(1, 9)]  # 8 Mata Pelajaran
ruangan_list = ["Lab 1", "Lab 2", "Lab 3"]      # 3 Ruangan
waktu_list = ["Slot 1", "Slot 2", "Slot 3", "Slot 4"] # 4 Slot Waktu

# 1. Fungsi initiateSolusiAwal
def initiateSolusiAwal(mapel, ruangan, waktu):
    solusi = {}
    for m in mapel:
        solusi[m] = (random.choice(ruangan), random.choice(waktu))
    return solusi

# 2. Fungsi hitungBentrok
def hitungBentrok(solusi):
    slot_terpakai = {}
    for r_w in solusi.values():
        slot_terpakai[r_w] = slot_terpakai.get(r_w, 0) + 1
    
    # Hitung jumlah bentrok (slot yang diisi > 1 mapel)
    bentrok = sum(count - 1 for count in slot_terpakai.values() if count > 1)
    return bentrok

# 3. Fungsi cariTetangga
def cariTetangga(solusi_saat_ini):
    tetangga = []
    for mapel in solusi_saat_ini.keys():
        for r in ruangan_list:
            for w in waktu_list:
                # Lewati jika sama dengan kondisi saat ini
                if solusi_saat_ini[mapel] == (r, w):
                    continue
                # Salin dan ubah satu komponen
                solusi_baru = solusi_saat_ini.copy()
                solusi_baru[mapel] = (r, w)
                tetangga.append(solusi_baru)
    return tetangga

# 4. Menjalankan Algoritma Hill Climbing
def hill_climbing_jadwal():
    solusi_sekarang = initiateSolusiAwal(mapel_list, ruangan_list, waktu_list)
    bentrok_sekarang = hitungBentrok(solusi_sekarang)
    
    iterasi = 0
    while bentrok_sekarang > 0:
        daftar_tetangga = cariTetangga(solusi_sekarang)
        if not daftar_tetangga:
            break
            
        # Cari tetangga terbaik (bentrok terkecil)
        tetangga_terbaik = min(daftar_tetangga, key=hitungBentrok)
        bentrok_terbaik = hitungBentrok(tetangga_terbaik)
        
        # Jika tidak ada peningkatan, berhenti (local optima)
        if bentrok_terbaik >= bentrok_sekarang:
            break
            
        solusi_sekarang = tetangga_terbaik
        bentrok_sekarang = bentrok_terbaik
        iterasi += 1
        
    print("--- HILL CLIMBING SELESAI ---")
    print(f"Total Iterasi: {iterasi}")
    print(f"Jumlah Bentrok Akhir: {bentrok_sekarang}")
    print("Solusi Jadwal:", solusi_sekarang)

# Jalankan fungsi
hill_climbing_jadwal()

--- HILL CLIMBING SELESAI ---
Total Iterasi: 0
Jumlah Bentrok Akhir: 0
Solusi Jadwal: {'Mapel_1': ('Lab 1', 'Slot 2'), 'Mapel_2': ('Lab 3', 'Slot 2'), 'Mapel_3': ('Lab 2', 'Slot 1'), 'Mapel_4': ('Lab 2', 'Slot 2'), 'Mapel_5': ('Lab 1', 'Slot 4'), 'Mapel_6': ('Lab 1', 'Slot 3'), 'Mapel_7': ('Lab 3', 'Slot 4'), 'Mapel_8': ('Lab 1', 'Slot 1')}


In [27]:
import networkx as nx

# 12. Menginisialisasi graf tidak berarah dan menambahkan node
g_bfs = nx.Graph()
nodes_rs = ['Pintu Masuk', 'Lobi', 'Apotek', 'Poli Umum', 'Radiologi', 'Ruang ICU']
g_bfs.add_nodes_from(nodes_rs)

# 13. Menambahkan edge berbobot 0
edges_bfs = [
    ('Pintu Masuk', 'Lobi', 0), ('Lobi', 'Apotek', 0), 
    ('Lobi', 'Poli Umum', 0), ('Apotek', 'Radiologi', 0),
    ('Poli Umum', 'Ruang ICU', 0), ('Radiologi', 'Ruang ICU', 0)
]
g_bfs.add_weighted_edges_from(edges_bfs)

# 14. Fungsi heuristik
def heuristable(a, b):
    # Estimasi jarak node ke 'Ruang ICU'
    tabel_heuristik = {
        'Pintu Masuk': 500, 'Lobi': 350, 'Apotek': 400,
        'Poli Umum': 150, 'Radiologi': 200, 'Ruang ICU': 0
    }
    return tabel_heuristik.get(a, 0)

# 15. Mencari jalur Greedy BFS menggunakan nx.astar_path
# Catatan: nx.astar_path bertindak menjadi Greedy BFS murni ketika bobot seluruh edge diatur ke 0.
jalur_bfs = nx.astar_path(g_bfs, source='Pintu Masuk', target='Ruang ICU', heuristic=heuristable, weight='weight')
total_bobot_bfs = nx.astar_path_length(g_bfs, source='Pintu Masuk', target='Ruang ICU', heuristic=heuristable, weight='weight')

print("\n--- GREEDY BEST-FIRST SEARCH ---")
print("Jalur yang dikunjungi:", " -> ".join(jalur_bfs))
print("Total Bobot Aktual (sesuai spesifikasi edge=0):", total_bobot_bfs)


--- GREEDY BEST-FIRST SEARCH ---
Jalur yang dikunjungi: Pintu Masuk -> Lobi -> Poli Umum -> Ruang ICU
Total Bobot Aktual (sesuai spesifikasi edge=0): 0


In [28]:
import networkx as nx

# 16. Membuat graf berarah berbobot, menambahkan node dan edge
g_astar = nx.DiGraph()
nodes_kota = ['Terminal Bus', 'Taman Kota', 'Mall', 'Stasiun', 'Balai Kota']
g_astar.add_nodes_from(nodes_kota)

edges_astar = [
    ('Terminal Bus', 'Taman Kota', 300),
    ('Terminal Bus', 'Mall', 500),
    ('Taman Kota', 'Stasiun', 200),
    ('Mall', 'Stasiun', 150),
    ('Stasiun', 'Balai Kota', 250),
    ('Taman Kota', 'Balai Kota', 700)
]
g_astar.add_weighted_edges_from(edges_astar)

# 17. Fungsi heuristik & menjalankan pencarian A*
def heuristable_astar(a, b):
    tabel_heuristik = {
        'Terminal Bus': 600, 'Taman Kota': 400, 'Mall': 350,
        'Stasiun': 200, 'Balai Kota': 0
    }
    return tabel_heuristik.get(a, 0)

jalur_astar = nx.astar_path(g_astar, source='Terminal Bus', target='Balai Kota', heuristic=heuristable_astar, weight='weight')
panjang_astar = nx.astar_path_length(g_astar, source='Terminal Bus', target='Balai Kota', heuristic=heuristable_astar, weight='weight')

print("\n--- A* SEARCH ---")
print("Jalur Terpendek:", " -> ".join(jalur_astar))
print("Total Lintasan Aktual (g(n)):", panjang_astar)


--- A* SEARCH ---
Jalur Terpendek: Terminal Bus -> Taman Kota -> Stasiun -> Balai Kota
Total Lintasan Aktual (g(n)): 750


In [29]:
import sympy as sp

# Konfigurasi agar tampilan rumus matematika tercetak rapi di Jupyter Notebook
sp.init_printing() 

# 18. Membuat simbol dan persamaan
x, y, z = sp.symbols('x y z')
ekspresi = 3*x**2 - 2*y + z
print("\n--- SYMPY ---")
print("18. Ekspresi Simbolik:")
sp.pprint(ekspresi)

# 19. Mengembangkan persamaan aljabar menggunakan .expand()
persamaan_kubik = (x + 2*y)**3
persamaan_expand = persamaan_kubik.expand()
print("\n19. Hasil Perkalian/Ekspansi Aljabar:")
sp.pprint(persamaan_expand)

# 20. Menghitung turunan persamaan menggunakan .diff()
fungsi_turunan = 5*x**4 - sp.Rational(2,3)*x + 7
hasil_turunan = fungsi_turunan.diff(x)
print("\n20. Hasil Turunan terhadap x:")
sp.pprint(hasil_turunan)

# 21. Melakukan operasi matriks
A = sp.Matrix([[3, 1], [2, 4]])
B = sp.Matrix([[1, 0], [5, 2]])

det_A = A.det()
inv_B = B.inv()
perkali_AB = A * B

print("\n21. Operasi Matriks:")
print("Determinan Matriks A:", det_A)
print("Invers Matriks B:")
sp.pprint(inv_B)
print("Hasil Perkalian A x B:")
sp.pprint(perkali_AB)


--- SYMPY ---
18. Ekspresi Simbolik:
   2          
3⋅x  - 2⋅y + z

19. Hasil Perkalian/Ekspansi Aljabar:
 3      2           2      3
x  + 6⋅x ⋅y + 12⋅x⋅y  + 8⋅y 

20. Hasil Turunan terhadap x:
    3   2
20⋅x  - ─
        3

21. Operasi Matriks:
Determinan Matriks A: 10
Invers Matriks B:
⎡ 1     0 ⎤
⎢         ⎥
⎣-5/2  1/2⎦
Hasil Perkalian A x B:
⎡8   2⎤
⎢     ⎥
⎣22  8⎦


In [30]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt

# 22. Menginisialisasi variabel Antecedent dan Consequent
food_quality = ctrl.Antecedent(np.arange(0, 11, 1), 'food_quality')
service_speed = ctrl.Antecedent(np.arange(0, 11, 1), 'service_speed')
tip = ctrl.Consequent(np.arange(0, 31, 1), 'tip')

# 23. Mendefinisikan fungsi keanggotaan triangular
food_quality['bad'] = fuzz.trimf(food_quality.universe, [0, 0, 5])
food_quality['average'] = fuzz.trimf(food_quality.universe, [3, 5, 8])
food_quality['good'] = fuzz.trimf(food_quality.universe, [6, 10, 10])

service_speed['slow'] = fuzz.trimf(service_speed.universe, [0, 0, 5])
service_speed['moderate'] = fuzz.trimf(service_speed.universe, [3, 5, 7])
service_speed['fast'] = fuzz.trimf(service_speed.universe, [5, 10, 10])

tip['low'] = fuzz.trimf(tip.universe, [0, 0, 15])
tip['medium'] = fuzz.trimf(tip.universe, [10, 15, 20])
tip['high'] = fuzz.trimf(tip.universe, [15, 30, 30])

# Menampilkan graf fungsi keanggotaan (Gunakan blok ini di lingkungan Jupyter)
food_quality.view()
service_speed.view()
tip.view()
plt.show()

# 24. Membuat aturan fuzzy menggunakan ctrl.Rule()
rule1 = ctrl.Rule(food_quality['good'] & service_speed['fast'], tip['high'])
rule2 = ctrl.Rule(food_quality['average'], tip['medium'])
rule3 = ctrl.Rule(food_quality['bad'] | service_speed['slow'], tip['low'])

# 25. Membangun sistem kontrol, menjalankan simulasi, dan menampilkan hasil
tip_ctrl = ctrl.ControlSystem([rule1, rule2, rule3])
tip_simulation = ctrl.ControlSystemSimulation(tip_ctrl)

# Memberikan nilai input
tip_simulation.input['food_quality'] = 7
tip_simulation.input['service_speed'] = 8

# Menjalankan komputasi
tip_simulation.compute()

# Mencetak hasil output tip dan visualisasi defuzzifikasi
print("\n--- SCIKIT-FUZZY ---")
print(f"Hasil Output Tip Hasil Defuzzifikasi: {tip_simulation.output['tip']:.4f}")

# Menampilkan visualisasi hasil defuzzifikasi tip
tip.view(sim=tip_simulation)
plt.show()

ImportError: `FuzzyVariableVisualizer` can only be used with `matplotlib` present in the system.